# Manual Transformer Structure  
**Author:** Shuxin Qian     
**Date:** 02-21-2026    
----

## Introduction
This notebook provides a manual implementation of the Transformer architecture, a powerful model for sequence-to-sequence tasks such as machine translation and natural language processing. The Transformer model, introduced by Vaswani et al. in 2017, relies on self-attention mechanisms to capture dependencies between input and output sequences without the need for recurrent or convolutional layers.

This work will use Pytorch, numpy, and matplotlib to build and visualize the components of the Transformer model. We will cover the following key components:       
1. **Multi-Head Attention**: A mechanism that allows the model to focus on different parts of the input sequence simultaneously.        
2. **Position-wise Feed-Forward Networks**: A fully connected feed-forward network applied to each position separately and identically.     
3. **Positional Encoding**: A technique to inject information about the position of tokens in the sequence, since the Transformer does not have any inherent notion of order.       
4. **Encoder and Decoder Layers**: The building blocks of the Transformer architecture, where the encoder processes the input sequence and the decoder generates the output sequence.       
5. **Training Loop**: A simple training loop to demonstrate how to train the Transformer model on a sample dataset.     
By the end of this notebook, you will have a clear understanding of how the Transformer architecture works and how to implement it from scratch.      
  
This notwwork will focus training and education purposes, so the most sturctures will be built by numpy, not from Pytorch.    Because the main purpose is to understand the structure of the Transformer, this file will not include the code for text processing, which is a crucial step in preparing data for the Transformer model. This file will more use pytorch functions to set up the training loop and testing loop, as well as visualization.       

This file focuses the training loop and testing loop, as well as visualization. Besides, there are other files for text processing, model structure.        

The total files include:
1. **Transformer.ipynb**: The main notebook that contains the implementation of the Transformer architecture, including the training loop and testing loop, as well as visualization.
2. **Text_Processing.ipynb**: A notebook dedicated to text preprocessing techniques, such as tokenization, padding, and creating vocabulary for the Transformer model.
3. **Position_Encoding.ipynb**: A notebook that explains and implements positional encoding, which is crucial for the Transformer to understand the order of tokens in a sequence.
4. **Multi_Head_Attention.ipynb**: A notebook that focuses on the implementation of the multi-head attention mechanism, which allows the model to attend to different parts of the input sequence simultaneously.
5. **Feed_Forward_Network.ipynb**: A notebook that implements the position-wise feed-forward networks used in the Transformer architecture.
6. **Encoder_Decoder_Layers.ipynb**: A notebook that builds the encoder and decoder layers of the Transformer, which are the core components of the architecture.
7. **Forward_Backward_Pass.ipynb**: A notebook that demonstrates the forward and backward pass of the Transformer model, including how to compute gradients and update parameters during training.
8. **Utils.ipynb**: it implements evaluation metrics commonly used for sequence-to-sequence tasks, such as BLEU score and ROUGE score, to assess the performance of the Transformer model. besides, it also includes mask generation functions for the Transformer model, such as padding masks and look-ahead masks, which are essential for training the model effectively.   
9. **Sample_Dataset.ipynb**: A notebook that creates a sample dataset for training the Transformer model, which can be used for demonstration purposes. This dataset will be simple and synthetic, designed to illustrate the training process of the Transformer without the need for complex data preprocessing.


In [1]:
import os
import torch
import torch.nn as nn
from Text_Processing import TextBedding, loading_data
from torch.utils.data import DataLoader, TensorDataset
from Forward_Backward_Pass import ForwardBackwardPass
from Utils import Loss_Plot
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
print(torch.__version__)
print(torch.backends.openmp.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"

[NbConvertApp] Converting notebook Utils.ipynb to script
[NbConvertApp] Writing 3829 bytes to Utils.py
2.10.0+cu128
True


In [ ]:
## Training Loop Setting by Pytorch
class Trainer:
    def __init__(self, model, dataloader, val_dataloader, optimizer, criterion, num_epochs, vocab_target, vocab_text, device):
        self.model = model
        self.train_dataloader = dataloader
        self.val_dataloader = val_dataloader
        self.optimizer = optimizer
        self.criterion = criterion
        self.num_epochs = num_epochs
        self.vocab_target = vocab_target
        self.vocab_text = vocab_text
        self.device = device
        self.loss_plot = Loss_Plot()    
    def train_model(self):
        sos_id = self.vocab_target['<sos>']  # 2
        eos_id = self.vocab_target['<eos>']  # 3
        for epoch in range(self.num_epochs):
            print(f"Epoch {epoch + 1}")
            total_loss = 0
            for batch_idx, batch in enumerate(self.train_dataloader):
                if (batch_idx+1) %10 ==0:
                    print(f"Processing batch {batch_idx + 1}/{len(self.train_dataloader)}")
                    print(f"Batch {batch_idx + 1} Loss: {loss.item():.4f}")
                textid_batch, targetid_batch = batch
                textid_batch, targetid_batch = textid_batch.to(self.device), targetid_batch.to(self.device)

                self.optimizer.zero_grad()
                ## output sample:(SOS, w1, w2, w3, EOS, PAD)
                ## input to decoder:(SOS, w1, w2, w3, EOS)
                ## target for loss:(w1, w2, w3, EOS, PAD)
                output = self.model.forward(textid_batch, targetid_batch[:, :-1], train_language="English", target_language="German")
                loss = self.criterion(output.view(-1, output.size(-1)), targetid_batch[:, 1:].contiguous().view(-1))

                loss.backward()
                self.optimizer.step()
                ## Accumulate loss for plotting
                total_loss += loss.item()
                avg_loss = total_loss / (batch_idx + 1)
                val_loss = self.validate_model()
                self.loss_plot.update(avg_loss, val_loss)
            if (epoch+1)%5==0:
                self.save_model()
    def validate_model(self):
        self.model.eval()
        total_loss = 0
        with torch.no_grad():
            for batch_idx, batch in enumerate(self.val_dataloader):
                textid_batch, targetid_batch = batch
                textid_batch, targetid_batch = textid_batch.to(self.device), targetid_batch.to(self.device)
                output = self.model.forward(textid_batch, targetid_batch[:, :-1], train_language="English", target_language="German")
                loss = self.criterion(output.view(-1, output.size(-1)), targetid_batch[:, 1:].contiguous().view(-1))
                total_loss += loss.item()
        avg_loss = total_loss / len(self.val_dataloader)
        return avg_loss
    def plot(self):
        print(self.loss_plot.plot())
    def save_model(self, path="transformer_checkpoint.pth"):
        checkpoint = {
            "model_state_dict": self.model.state_dict(),
            "optimizer_state_dict": self.optimizer.state_dict(),
            "vocab_text": self.vocab_text,
            "vocab_target": self.vocab_target,
            "train_losses": self.loss_plot.train_loss_history,
            "val_losses": self.loss_plot.val_loss_history}
        torch.save(checkpoint, path)
        print(f"Model saved to {path}")

if __name__ == "__main__":
    # Define hyperparameters
    n_epoch=30
    batch_size = 64 # Define your batch size
    embedding_dim = 128
    d_model = 128
    nhead = 4
    num_encoder_layers = 6
    num_decoder_layers = 6
    dim_feedforward = 512
    max_seq_length_text = 39
    max_seq_length_target = 41
    # Load data
    train_multi30k_df, vaild_multi30k_df, test_multi30k_df = loading_data()
    
    Embed_train=TextBedding(embedding_dim=embedding_dim)
    vocab_text = Embed_train.build_vocab(train_multi30k_df[:,0], language="English")  # Build vocab for source text
    vocab_target = Embed_train.build_vocab(train_multi30k_df[:,1], language="German")  # Build vocab for target text
    all_token_ids_text = Embed_train.text_to_ids(train_multi30k_df[:,0], language="English")
    all_token_ids_target = Embed_train.text_to_ids(train_multi30k_df[:,1], language="German")
    token_id_valid_text = Embed_train.text_to_ids(vaild_multi30k_df[:,0], language="English")
    token_id_valid_target = Embed_train.text_to_ids(vaild_multi30k_df[:,1], language="German")
    Embed_train.initialize_embedding_layer(language="English")
    Embed_train.initialize_embedding_layer(language="German")
    Embed_train.embedding_layer["English"] = Embed_train.embedding_layer["English"].to(device)
    Embed_train.embedding_layer["German"] = Embed_train.embedding_layer["German"].to(device)




    Transformer_model=ForwardBackwardPass(embedding_train=Embed_train.embedding_layer,
                                           output_size=len(vocab_target), d_model=embedding_dim, nhead=nhead, 
                                           num_encoder_layers=num_encoder_layers, num_decoder_layers=num_decoder_layers, 
                                           dim_feedforward=dim_feedforward, max_seq_length_text=max_seq_length_text, max_seq_length_target=max_seq_length_target).to(device)
    print(next(Embed_train.embedding_layer["English"].parameters()).device)
    print(next(Transformer_model.parameters()).device)

    dataset = TensorDataset(all_token_ids_text, all_token_ids_target) 
    val_dataset = TensorDataset(token_id_valid_text, token_id_valid_target)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    optimizer = torch.optim.Adam(Transformer_model.parameters(), lr=0.001) # Define your optimizer
    criterion = nn.CrossEntropyLoss(ignore_index=vocab_target['<pad>']) # Define your loss function
    loss_plot=Loss_Plot()
    model_trainer=Trainer(model=Transformer_model, dataloader=dataloader, val_dataloader=val_dataloader, optimizer=optimizer, criterion=criterion, num_epochs=n_epoch, vocab_target=vocab_target, vocab_text=vocab_text, device=device)
    model_trainer.train_model()
 

cuda:0
cuda:0
Epoch 1
Processing batch 1/454
Batch 1 Loss: 81.6482
Processing batch 2/454
Batch 2 Loss: 70.4353
Processing batch 3/454
Batch 3 Loss: 64.0437
Processing batch 4/454
Batch 4 Loss: 60.2188
Processing batch 5/454
Batch 5 Loss: 59.9208
Processing batch 6/454
Batch 6 Loss: 57.4166
Processing batch 7/454
Batch 7 Loss: 51.9369
Processing batch 8/454
Batch 8 Loss: 49.6505
Processing batch 9/454
Batch 9 Loss: 45.1515
Processing batch 10/454
Batch 10 Loss: 43.7766
Processing batch 11/454
Batch 11 Loss: 41.6949
Processing batch 12/454
Batch 12 Loss: 40.1494
Processing batch 13/454
Batch 13 Loss: 38.8606
Processing batch 14/454
Batch 14 Loss: 35.2460
Processing batch 15/454
Batch 15 Loss: 33.0994
Processing batch 16/454
Batch 16 Loss: 30.7864
Processing batch 17/454
Batch 17 Loss: 29.6066
Processing batch 18/454
Batch 18 Loss: 25.9803
Processing batch 19/454
Batch 19 Loss: 24.2040
Processing batch 20/454
Batch 20 Loss: 22.8911
Processing batch 21/454
Batch 21 Loss: 20.5187
Processin

Processing batch 175/454
Batch 175 Loss: 7.6317
Processing batch 176/454
Batch 176 Loss: 7.6839
Processing batch 177/454
Batch 177 Loss: 7.6702
Processing batch 178/454
Batch 178 Loss: 7.6972
Processing batch 179/454
Batch 179 Loss: 7.5420
Processing batch 180/454
Batch 180 Loss: 7.6435
Processing batch 181/454
Batch 181 Loss: 7.5573
Processing batch 182/454
Batch 182 Loss: 7.4934
Processing batch 183/454
Batch 183 Loss: 7.4180
Processing batch 184/454
Batch 184 Loss: 7.5423
Processing batch 185/454
Batch 185 Loss: 7.4891
Processing batch 186/454
Batch 186 Loss: 7.6092
Processing batch 187/454
Batch 187 Loss: 7.6478
Processing batch 188/454
Batch 188 Loss: 7.5754
Processing batch 189/454
Batch 189 Loss: 7.4754
Processing batch 190/454
Batch 190 Loss: 7.5414
Processing batch 191/454
Batch 191 Loss: 7.6370
Processing batch 192/454
Batch 192 Loss: 7.6950
Processing batch 193/454
Batch 193 Loss: 7.4486
Processing batch 194/454
Batch 194 Loss: 7.4750
Processing batch 195/454
Batch 195 Loss:

Processing batch 346/454
Batch 346 Loss: 8.9715
Processing batch 347/454
Batch 347 Loss: 8.0730
Processing batch 348/454
Batch 348 Loss: 7.6363
Processing batch 349/454
Batch 349 Loss: 7.2300
Processing batch 350/454
Batch 350 Loss: 9.5878
Processing batch 351/454
Batch 351 Loss: 8.9288
Processing batch 352/454
Batch 352 Loss: 10.2509
Processing batch 353/454
Batch 353 Loss: 10.9428
Processing batch 354/454
Batch 354 Loss: 10.5573
Processing batch 355/454
Batch 355 Loss: 11.0167
Processing batch 356/454
Batch 356 Loss: 10.6250
Processing batch 357/454
Batch 357 Loss: 10.6580
Processing batch 358/454
Batch 358 Loss: 10.8036
Processing batch 359/454
Batch 359 Loss: 10.6136
Processing batch 360/454
Batch 360 Loss: 10.3545
Processing batch 361/454
Batch 361 Loss: 11.5695
Processing batch 362/454
Batch 362 Loss: 11.0168
Processing batch 363/454
Batch 363 Loss: 11.0030
Processing batch 364/454
Batch 364 Loss: 10.8806
Processing batch 365/454
Batch 365 Loss: 11.6961
Processing batch 366/454
B

Processing batch 66/454
Batch 66 Loss: 6.5900
Processing batch 67/454
Batch 67 Loss: 6.4820
Processing batch 68/454
Batch 68 Loss: 6.5077
Processing batch 69/454
Batch 69 Loss: 6.4722
Processing batch 70/454
Batch 70 Loss: 6.3706
Processing batch 71/454
Batch 71 Loss: 6.7011
Processing batch 72/454
Batch 72 Loss: 6.4057
Processing batch 73/454
Batch 73 Loss: 6.3430
Processing batch 74/454
Batch 74 Loss: 6.4787
Processing batch 75/454
Batch 75 Loss: 6.5588
Processing batch 76/454
Batch 76 Loss: 6.3543
Processing batch 77/454
Batch 77 Loss: 6.6049
Processing batch 78/454
Batch 78 Loss: 6.6223
Processing batch 79/454
Batch 79 Loss: 6.4387
Processing batch 80/454
Batch 80 Loss: 6.5910
Processing batch 81/454
Batch 81 Loss: 6.4887
Processing batch 82/454
Batch 82 Loss: 6.5598
Processing batch 83/454
Batch 83 Loss: 6.3814
Processing batch 84/454
Batch 84 Loss: 6.5727
Processing batch 85/454
Batch 85 Loss: 6.3839
Processing batch 86/454
Batch 86 Loss: 6.3912
Processing batch 87/454
Batch 87 L

Processing batch 239/454
Batch 239 Loss: 6.4462
Processing batch 240/454
Batch 240 Loss: 6.3046
Processing batch 241/454
Batch 241 Loss: 6.3324
Processing batch 242/454
Batch 242 Loss: 6.4072
Processing batch 243/454
Batch 243 Loss: 6.5261
Processing batch 244/454
Batch 244 Loss: 6.3248
Processing batch 245/454
Batch 245 Loss: 6.3858
Processing batch 246/454
Batch 246 Loss: 6.5427
Processing batch 247/454
Batch 247 Loss: 6.3649
Processing batch 248/454
Batch 248 Loss: 6.5247
Processing batch 249/454
Batch 249 Loss: 6.3079
Processing batch 250/454
Batch 250 Loss: 6.3577
Processing batch 251/454
Batch 251 Loss: 6.2562
Processing batch 252/454
Batch 252 Loss: 6.5040
Processing batch 253/454
Batch 253 Loss: 6.5399
Processing batch 254/454
Batch 254 Loss: 6.3669
Processing batch 255/454
Batch 255 Loss: 6.3420
Processing batch 256/454
Batch 256 Loss: 6.3610
Processing batch 257/454
Batch 257 Loss: 6.2824
Processing batch 258/454
Batch 258 Loss: 6.3012
Processing batch 259/454
Batch 259 Loss:

...

Processing batch 98/454
Batch 98 Loss: 2.5167
Processing batch 99/454
Batch 99 Loss: 2.5202
Processing batch 100/454
Batch 100 Loss: 2.1930
Processing batch 101/454
Batch 101 Loss: 2.3359
Processing batch 102/454
Batch 102 Loss: 2.1441
Processing batch 103/454
Batch 103 Loss: 2.4077
Processing batch 104/454
Batch 104 Loss: 2.3700
Processing batch 105/454
Batch 105 Loss: 2.2254
Processing batch 106/454
Batch 106 Loss: 2.4892
Processing batch 107/454
Batch 107 Loss: 2.3226
Processing batch 108/454
Batch 108 Loss: 2.2947
Processing batch 109/454
Batch 109 Loss: 2.5922
Processing batch 110/454
Batch 110 Loss: 2.1437
Processing batch 111/454
Batch 111 Loss: 2.6479
Processing batch 112/454
Batch 112 Loss: 2.3424
Processing batch 113/454
Batch 113 Loss: 2.3557
Processing batch 114/454
Batch 114 Loss: 2.6195
Processing batch 115/454
Batch 115 Loss: 2.5525
Processing batch 116/454
Batch 116 Loss: 2.2587
Processing batch 117/454
Batch 117 Loss: 2.5163
Processing batch 118/454
Batch 118 Loss: 2.5

Processing batch 269/454
Batch 269 Loss: 2.4258
Processing batch 270/454
Batch 270 Loss: 2.7654
Processing batch 271/454
Batch 271 Loss: 2.3126
Processing batch 272/454
Batch 272 Loss: 2.3063
Processing batch 273/454
Batch 273 Loss: 2.4972
Processing batch 274/454
Batch 274 Loss: 2.5753
Processing batch 275/454
Batch 275 Loss: 2.6280
Processing batch 276/454
Batch 276 Loss: 2.2984
Processing batch 277/454
Batch 277 Loss: 2.2940
Processing batch 278/454
Batch 278 Loss: 2.6179
Processing batch 279/454
Batch 279 Loss: 2.0925
Processing batch 280/454
Batch 280 Loss: 2.1182
Processing batch 281/454
Batch 281 Loss: 2.4961
Processing batch 282/454
Batch 282 Loss: 2.5521
Processing batch 283/454
Batch 283 Loss: 2.4983
Processing batch 284/454
Batch 284 Loss: 2.7368
Processing batch 285/454
Batch 285 Loss: 2.6290
Processing batch 286/454
Batch 286 Loss: 2.3697
Processing batch 287/454
Batch 287 Loss: 2.6459
Processing batch 288/454
Batch 288 Loss: 2.3097
Processing batch 289/454
Batch 289 Loss:

Processing batch 440/454
Batch 440 Loss: 2.5097
Processing batch 441/454
Batch 441 Loss: 2.6305
Processing batch 442/454
Batch 442 Loss: 2.3718
Processing batch 443/454
Batch 443 Loss: 2.1672
Processing batch 444/454
Batch 444 Loss: 2.2013
Processing batch 445/454
Batch 445 Loss: 2.3018
Processing batch 446/454
Batch 446 Loss: 1.9946
Processing batch 447/454
Batch 447 Loss: 2.3965
Processing batch 448/454
Batch 448 Loss: 2.5865
Processing batch 449/454
Batch 449 Loss: 2.2567
Processing batch 450/454
Batch 450 Loss: 2.8497
Processing batch 451/454
Batch 451 Loss: 2.3732
Processing batch 452/454
Batch 452 Loss: 2.6519
Processing batch 453/454
Batch 453 Loss: 2.5618
Processing batch 454/454
Batch 454 Loss: 2.1444
Epoch 25
Processing batch 1/454
Batch 1 Loss: 2.1748
Processing batch 2/454
Batch 2 Loss: 2.1554
Processing batch 3/454
Batch 3 Loss: 2.2983
Processing batch 4/454
Batch 4 Loss: 2.5050
Processing batch 5/454
Batch 5 Loss: 2.2204
Processing batch 6/454
Batch 6 Loss: 2.6299
Process

Processing batch 161/454
Batch 161 Loss: 2.2280
Processing batch 162/454
Batch 162 Loss: 2.3389
Processing batch 163/454
Batch 163 Loss: 2.2694
Processing batch 164/454
Batch 164 Loss: 2.2576
Processing batch 165/454
Batch 165 Loss: 2.4828
Processing batch 166/454
Batch 166 Loss: 2.3235
Processing batch 167/454
Batch 167 Loss: 2.3737
Processing batch 168/454
Batch 168 Loss: 2.4815
Processing batch 169/454
Batch 169 Loss: 2.6376
Processing batch 170/454
Batch 170 Loss: 2.3461
Processing batch 171/454
Batch 171 Loss: 2.2061
Processing batch 172/454
Batch 172 Loss: 2.5533
Processing batch 173/454
Batch 173 Loss: 2.3093
Processing batch 174/454
Batch 174 Loss: 2.3084
Processing batch 175/454
Batch 175 Loss: 2.4624
Processing batch 176/454
Batch 176 Loss: 2.2521
Processing batch 177/454
Batch 177 Loss: 2.5191
Processing batch 178/454
Batch 178 Loss: 2.5012
Processing batch 179/454
Batch 179 Loss: 2.3905
Processing batch 180/454
Batch 180 Loss: 2.2814
Processing batch 181/454
Batch 181 Loss:

Processing batch 332/454
Batch 332 Loss: 2.3020
Processing batch 333/454
Batch 333 Loss: 2.4288
Processing batch 334/454
Batch 334 Loss: 2.2976


In [ ]:
model_trainer.plot()
